In [2]:
import numpy as np
import ast
import pandas as pd
from plotnine import (
    ggplot, aes, geom_line, geom_point, geom_errorbar, facet_wrap, guides, scale_linetype_manual, 
    labs, labeller,theme_classic, theme, element_text, geom_vline, scale_y_continuous, geom_hline, 
    scale_x_continuous, scale_color_manual, ggtitle, scale_shape_manual, position_dodge, geom_smooth, 
    geom_text, scale_shape_discrete, scale_x_discrete
)

## Load and preprocess data

In [137]:
# Plotting parameters
dodge = position_dodge(width=0.6)
stimColors = {'Rule': '#715cad', 'Exception': '#6fa8dc'}

# Load the raw data
df = pd.read_csv('data/experiment2_raw.csv', dtype={'stimuli': str})

In [138]:
df

,subjectId,rt,correctResponse,stimuli,catch,timeout,taskPart,responseCoded,correct,trialNum,blockNum,stimType,curriculum,pretraining,stimulusRandomization,session,simple_rule_match,curriculum_label
0,0,2602.0,1,101,False,False,metalearning,0.0,0,0.0,0.0,Rule,No Curriculum,Simple,"[{'color': 'red', 'randomization': {'featureAs...",session_1,False,No \nCurriculum
1,0,1962.0,1,101,False,False,metalearning,1.0,1,1.0,0.0,Rule,No Curriculum,Simple,"[{'color': 'red', 'randomization': {'featureAs...",session_1,False,No \nCurriculum
2,0,1055.0,0,011,False,False,metalearning,0.0,1,2.0,0.0,Rule,No Curriculum,Simple,"[{'color': 'red', 'randomization': {'featureAs...",session_1,False,No \nCurriculum
3,0,1791.0,1,100,False,False,metalearning,1.0,1,3.0,0.0,Exception,No Curriculum,Simple,"[{'color': 'red', 'randomization': {'featureAs...",session_1,False,No \nCurriculum
4,0,929.0,1,101,False,False,metalearning,1.0,1,4.0,0.0,Rule,No Curriculum,Simple,"[{'color': 'red', 'randomization': {'featureAs...",session_1,False,No \nCurriculum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509323,1110,1143.0,0,001,False,False,test,1.0,0,11.0,5.0,Rule,No Curriculum,Simple,"[{'color': 'blue', 'randomization': {'featureA...",session_5,False,No \nCurriculum
509324,1110,2005.0,0,101,False,False,test,0.0,1,12.0,5.0,Rule,No Curriculum,Simple,"[{'color': 'blue', 'randomization': {'featureA...",session_5,False,No \nCurriculum
509325,1110,1578.0,0,110,False,False,test,0.0,1,13.0,5.0,Exception,No Curriculum,Simple,"[{'color': 'blue', 'randomization': {'featureA...",session_5,False,No \nCurriculum
509326,1110,3641.0,0,000,False,False,test,0.0,1,14.0,5.0,Rule,No Curriculum,Simple,"[{'color': 'blue', 'randomization': {'featureA...",session_5,False,No \nCurriculum


In [139]:
# Filter out catch trials
df = df[(df['catch'] == False)]

# Create a new column for curriculum labels for better readability in plots
label_map = {
    "No Curriculum": "No \nCurriculum",
    "Simple To Complex": "Simple \nto Complex",
}
df["curriculum_label"] = df["curriculum"].map(label_map)

In [140]:
#  Count unique participants per curriculum (uses existing `df`)
n_per_curr = df.groupby(['curriculum', 'pretraining'])['subjectId'].nunique().reset_index(name='n')
print(n_per_curr)
print("Total participants:", n_per_curr['n'].sum())

          curriculum pretraining    n
0      No Curriculum      Simple  266
1      No Curriculum         XOR  254
2  Simple To Complex      Simple  259
3  Simple To Complex         XOR  248
Total participants: 1027


In [141]:
#  Count unique participants per curriculum (uses existing `df`)
n_per_curr = df.groupby(['session'])['subjectId'].nunique().reset_index(name='n')
print(n_per_curr)
print("Total participants:", n_per_curr['n'].sum())

     session    n
0  session_1   96
1  session_2  116
2  session_3  416
3  session_4   78
4  session_5  321
Total participants: 1027


In [142]:
# Compute learning, pretraining and test dfs with totaltrialnum and bins

# Pretraining df
bin_size = 24
df_pretraining = df[(df['taskPart'] == 'metalearning')]
df_pretraining['totalTrialNum'] = (
    df_pretraining
      .sort_values(['subjectId', 'blockNum', 'trialNum'])
      .groupby('subjectId')
      .cumcount() + 1
)
df_pretraining = df_pretraining[df_pretraining['timeout'] == False]
df_pretraining['bin'] = ((df_pretraining['totalTrialNum'] - 1) // bin_size) + 1
df_pretraining['bin_mid'] = (df_pretraining['bin'] * bin_size - bin_size / 2).astype(float)

# Learning df
bin_size = 24
df_learning = df[df['taskPart'] == 'learning']
df_learning['totalTrialNum'] = (
    df_learning
      .sort_values(['subjectId', 'blockNum', 'trialNum'])
      .groupby('subjectId')
      .cumcount() + 1
)
df_learning = df_learning[df_learning['timeout'] == False]
df_learning['bin'] = ((df_learning['totalTrialNum'] - 1) // bin_size) + 1
# Bin mid point for plotting (use trial number in x-axis)
df_learning['bin_mid'] = (df_learning['bin'] * bin_size - bin_size / 2).astype(float)
df_learning_curriculum = df_learning[df_learning['curriculum'] == 'Simple To Complex'].to_csv('data/exp2_learning_curriculum.csv', index=False)
df_learning_control = df_learning[df_learning['curriculum'] == 'No Curriculum'].to_csv('data/exp2_learning_control.csv', index=False)

# Test df
test_df = df[(df['taskPart'] == 'test') & (df['timeout'] == False)]
test_df[['stimType','correct','pretraining','curriculum','trialNum','subjectId']].to_csv('data/exp2_test.csv', index=False)

/var/folders/1t/nrvh1_812fq_4bhspd0mgxlm0000gr/T/ipykernel_56006/2383735690.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/var/folders/1t/nrvh1_812fq_4bhspd0mgxlm0000gr/T/ipykernel_56006/2383735690.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


## Pretraining performance

In [212]:
# Per-subject accuracy per bin
per_sub = (
    df_pretraining
    .groupby(['pretraining', 'subjectId', 'bin', 'bin_mid'], as_index=False)
    .agg(acc=('correct', 'mean'))
)

# Across-subject summary: mean and standard error (SE)
summary = (
    per_sub
    .groupby(['pretraining', 'bin', 'bin_mid'], as_index=False)
    .agg(mean_acc=('acc', 'mean'), sem=('acc', lambda x: x.std(ddof=1) / np.sqrt(len(x))))
)
summary['ymin'] = summary['mean_acc'] - summary['sem']
summary['ymax'] = summary['mean_acc'] + summary['sem']

max_trial = int(df_pretraining['totalTrialNum'].max())

# Build the plot with individual subject lines
p = (
    ggplot(summary, aes(x='bin_mid', y='mean_acc', shape='pretraining'))
    + geom_line(size=1)
    + geom_point(size=2)
    + geom_errorbar(aes(ymin='ymin', ymax='ymax'), width=6)
    + geom_hline(yintercept=0.5, linetype='dashed', color='black', alpha=1)
    + labs(x='Trial', y='Accuracy')
    + scale_x_continuous(limits=(0, max_trial))
    + scale_y_continuous(limits=(0.5, 1))
    + theme_classic()
    + labs(color='')
    + theme(axis_text=element_text(size=7), text=element_text(size=10), figure_size=(2,2),legend_position='none')
)

p

p.save(
    filename='analysis/figs/fig4E.svg',
    format="svg",
    verbose=False
)

In [204]:
# Per-subject accuracy per bin
per_sub_learning = (
    df_learning
    .groupby(['curriculum_label','pretraining','subjectId','stimType', 'bin', 'bin_mid'], as_index=False)
    .agg(acc=('correct', 'mean'))
)

# Across-subject summary: mean and standard error (SE)
summary = (
    per_sub_learning
    .groupby(['curriculum_label', 'pretraining', 'stimType', 'bin', 'bin_mid'], as_index=False)
    .agg(mean_acc=('acc', 'mean'), sem=('acc', lambda x: x.std(ddof=1) / np.sqrt(len(x))))
)
# Create ymin/ymax columns for errorbars
summary['ymin'] = summary['mean_acc'] - summary['sem']
summary['ymax'] = summary['mean_acc'] + summary['sem']

# Compute vertical lines every 48 trials (in trial number space)
max_trial = int(df_learning['totalTrialNum'].max())
vlines = list(range(48, max_trial, 48))
# Put vlines into a small DataFrame so plotnine treats them as continuous values
vline_df = pd.DataFrame({'x': vlines}) if len(vlines) > 0 else pd.DataFrame({'x': []})

# Build the plot
p = (
    ggplot(summary, aes(x='bin_mid', y='mean_acc', color='stimType', shape='pretraining'))
    + geom_line()
    + geom_point()
    + geom_errorbar(aes(ymin='ymin', ymax='ymax'), width=6)
    + facet_wrap('~curriculum_label')
    + (geom_vline(mapping=aes(xintercept='x'), data=vline_df, color='red', linetype='dashed', alpha=0.2) if not vline_df.empty else 0)
    + geom_hline(yintercept=0.5, linetype='solid', color='black', alpha=1)
    + labs(x='Trial', y='Accuracy', shape='Pretraining', color='Stimulus Type')
    + scale_x_continuous(limits=(0, max_trial))
    + scale_y_continuous(limits=(0, 1))
    + scale_color_manual(values=stimColors)
    + theme_classic()
    + theme(axis_text=element_text(size=7), text=element_text(size=10),figure_size=(2.5,2), panel_spacing_x=0.04, legend_position='none')
)

p.save(
    filename='analysis/figs/fig4F.svg',
    format="svg",
    verbose=False
)


In [211]:
# Per-subject accuracy in the test phase per curriculum and stimType
per_sub_test = (
    test_df
    .groupby(['curriculum_label', 'pretraining', 'subjectId', 'stimType'], as_index=False)
    .agg(acc=('correct', 'mean'))
)

# Across-subject summary: mean and SE
summary_test = (
    per_sub_test
    .groupby(['curriculum_label', 'pretraining', 'stimType'], as_index=False)
    .agg(mean_acc=('acc', 'mean'), sem=('acc', lambda x: x.std(ddof=1) / np.sqrt(len(x))))
)
summary_test['ymin'] = summary_test['mean_acc'] - summary_test['sem']
summary_test['ymax'] = summary_test['mean_acc'] + summary_test['sem']

# Plot: curriculum on x-axis, accuracy on y-axis, stimType side-by-side
p_test = (
    ggplot(summary_test, aes(x='curriculum_label', y='mean_acc', color='stimType', shape='pretraining'))
    + geom_point(position=dodge, size=2)
    + geom_errorbar(aes(ymin='ymin', ymax='ymax'), width=0.2, position=dodge)
    + labs(x='Curriculum', y='Accuracy', color='Stimulus type', shape='Pretraining')
    + scale_y_continuous(limits=(0.5, 1))
    + scale_x_discrete(labels={'simple_with_noise': 'Simple\n Pretraining', 'xor': 'XOR\n Pretraining'})
    + scale_color_manual(values=stimColors)
    + theme_classic()
    + geom_hline(yintercept=0.5, linetype='dashed', color='black', alpha=1)
    # make x tick labels larger and not tilted
    + theme(axis_text=element_text(size=7), text=element_text(size=10),figure_size=(2.2,2), legend_text=element_text(size=7), legend_key_height=8, legend_position='none')
)

p_test.save(
    filename='analysis/figs/fig4G.svg',
    format="svg",
    verbose=False
)


## SMRT (switch / match RT ratio) and selective attention analyses
For each consecutive pair of trials within (subjectId, blockNum), each of the 3 stimulus features (positions 0,1,2) is either "match" (same value as the previous trial) or "switch". Within bins of `bin_size` trials, we average RT separately for match vs switch trials for each feature index and compute:
    SMRT_i = mean RT(match_i) / mean RT(switch_i)
    selective_attention_i = SMRT_i - mean(SMRT_total)

A greater selective_attention_i means switching feature i slows the subject down more than switching the other features -> they are attending to feature i.

In [146]:
bin_size = 54

# Pair each trial with the immediately preceding trial in the SAME block
df_smrt_src = df_pretraining.sort_values(['subjectId', 'trialNum']).reset_index(drop=True)
df_smrt_src['prev_stimuli'] = df_smrt_src.groupby(['subjectId', 'blockNum'])['stimuli'].shift(1)
df_smrt_src['prev_trialNum'] = df_smrt_src.groupby(['subjectId', 'blockNum'])['trialNum'].shift(1)
df_smrt_src['prev_correct'] = df_smrt_src.groupby(['subjectId', 'blockNum'])['correct'].shift(1)

# Keep only pairs that are truly consecutive trialNums (drops pairs straddling a filtered
# catch / timeout trial, and the first trial of each block which has no predecessor).
pair_df = df_smrt_src[df_smrt_src['trialNum'] - df_smrt_src['prev_trialNum'] == 1].copy()

# Drop pairs where first trial was incorrect (error corrective feedback means second trial is exactly the same as the first trial, so we don't want to include those in SMRT)
pair_df = pair_df[pair_df['prev_correct'] == 1].copy()

# Long form: one row per (trial, feature_idx)
n_features = 3
pieces = []
for idx in range(n_features):
    sub = pair_df[['subjectId', 'curriculum', 'pretraining', 'simple_rule_match',
                   'blockNum', 'trialNum', 'rt', 'stimuli', 'prev_stimuli', 'correct']].copy()
    sub['feature_idx'] = idx
    sub['transition'] = np.where(
        sub['stimuli'].str[idx] == sub['prev_stimuli'].str[idx],
        'match', 'switch'
    )
    pieces.append(sub)
smrt_long = pd.concat(pieces, ignore_index=True)

# Bin trial pairs by the cumulative trialNum
smrt_long['bin'] = (smrt_long['trialNum'] // bin_size).astype(int) + 1
smrt_long['bin_mid'] = (smrt_long['bin'] * bin_size - bin_size / 2).astype(float)

# Mean RT per (subject x bin x feature_idx x transition)
rt_means = (
    smrt_long
    .groupby(['subjectId', 'bin', 'bin_mid',
              'feature_idx', 'transition'], as_index=False)
    .agg(mean_rt=('rt', 'mean'))
)

# Pivot match/switch into columns, then compute SMRT
smrt_df = rt_means.pivot_table(
    index=['subjectId', 'bin', 'bin_mid', 'feature_idx'],
    columns='transition',
    values='mean_rt'
).reset_index()
smrt_df.columns.name = None
smrt_df = smrt_df.dropna(subset=['match', 'switch'])
smrt_df['SMRT'] = smrt_df['switch'] / smrt_df['match']

# Selective attention within each (subject, bin): SMRT_i divided by the mean SMRT across features
smrt_df['mean_SMRT'] = smrt_df.groupby(['subjectId', 'bin'])['SMRT'].transform('mean')
smrt_df['selective_attention'] = smrt_df['SMRT'] / smrt_df['mean_SMRT'] 

# Mean accuracy per (subject, bin) using all surviving trials in df_smrt_src (not just the
# paired-trial subset), then broadcast onto each feature_idx row of smrt_df.
df_smrt_src['bin'] = (df_smrt_src['trialNum'] // bin_size).astype(int) + 1
bin_acc = (
    df_smrt_src
    .groupby(['subjectId', 'pretraining', 'simple_rule_match', 'bin'], as_index=False)
    .agg(mean_correct=('correct', 'mean'))
)
smrt_df = smrt_df.merge(bin_acc, on=['subjectId', 'bin'], how='left')

In [168]:
# Plot SMRT over trial bins, by feature index
df_plot = smrt_df[smrt_df['pretraining'] == 'Simple']

summary_smrt = (
    df_plot
    .groupby(['pretraining', 'bin', 'bin_mid', 'feature_idx', ], as_index=False)
    .agg(
        mean_smrt=('SMRT', 'mean'),
        se_smrt=('SMRT', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
        selective_attention=('selective_attention', 'mean'),
        se_sa=('selective_attention', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
    )
)
summary_smrt['feature_idx_str'] = summary_smrt['feature_idx'].astype(str)
summary_smrt['feature_type'] = np.where(
    summary_smrt['feature_idx'] == 0,
    'Relevant',
    'Irrelevant'
)
summary_smrt['feature_type'] = pd.Categorical(
    summary_smrt['feature_type'],
    categories=['Relevant', 'Irrelevant'],
    ordered=True
)
summary_smrt['ymin_smrt'] = summary_smrt['mean_smrt'] - summary_smrt['se_smrt']
summary_smrt['ymax_smrt'] = summary_smrt['mean_smrt'] + summary_smrt['se_smrt']
summary_smrt['ymin_sa'] = summary_smrt['selective_attention'] - summary_smrt['se_sa']
summary_smrt['ymax_sa'] = summary_smrt['selective_attention'] + summary_smrt['se_sa']
last_relevant = (
    summary_smrt[summary_smrt['feature_idx'] == 0]
    .sort_values('bin_mid')
    .tail(1)
)

p_smrt = (
    ggplot(summary_smrt, aes(x='bin_mid', y='selective_attention', linetype='feature_type', group='feature_idx'))
    + geom_line()
    + geom_point()
    + geom_point(
        data=last_relevant,
        size=2,
        stroke=1.5,
        shape='*'
    )
    + geom_errorbar(aes(ymin='ymin_sa', ymax='ymax_sa'), width=4)
    + geom_hline(yintercept=1, linetype='dotted', color='black', alpha=0.5)
    + labs(x='Trial', y='Selective attention',
           color='Feature index', linetype='')
    + theme_classic()
    + theme(axis_text_x=element_text(rotation=0, ha='center'))
    + theme(axis_text=element_text(size=7), text=element_text(size=10),figure_size=(2,2),legend_position='bottom',
        legend_text=element_text(size=7),
        legend_direction='horizontal',
        legend_key_width=10,
        legend_key_height=5,
      )

)

p_smrt

p_smrt.save(
    filename='analysis/figs/fig5B.svg',
    format="svg",
    verbose=False
)


/opt/anaconda3/envs/nlp/lib/python3.12/site-packages/plotnine/scales/scale_linetype.py:45: PlotnineWarning: Using linetype for an ordinal variable is not advised.


## Does selective attention (proxy for compression) in pretraining predict whether curricula benefits later?

In [148]:
# Look at selective attention in the last bin of simple pretraining, and how it relates to test accuracy
pretraining_sa = (
    smrt_df[(smrt_df['pretraining'] == 'Simple')
            & (smrt_df['feature_idx'] == 0)
            & (smrt_df['bin'] >= 5)]
    .groupby('subjectId')['selective_attention'].mean()
)
# Add overall pretraining accuracy as a competing predictor of test accuracy
overall_pretrain_acc = (
    df_pretraining.groupby('subjectId')['correct'].mean().rename('pretrain_acc')
)

sa_test = test_df.merge(pretraining_sa.rename('selective_attention'), on='subjectId')
sa_test = sa_test.merge(overall_pretrain_acc, on='subjectId')
sa_test.to_csv('data/exp2_test_smrt.csv', index=False)

In [167]:
# Bin selective attention into quintiles WITHIN each curriculum (subjects do not overlap
# between curricula), then plot mean test accuracy +/- SE as lines.
# Curriculum is shown via shape + linetype on a single panel; stimType via color.
n_quantiles = 5

sa_per_sub_test = (
    sa_test
    .groupby(['curriculum_label', 'pretraining', 'subjectId', 'stimType','selective_attention','pretrain_acc'], as_index=False)
    .agg(acc=('correct', 'mean'))
)
sa_per_sub_test['sa_quantile'] = (
    sa_per_sub_test
    .groupby('curriculum_label')['selective_attention']
    .transform(lambda x: pd.qcut(x, q=n_quantiles, labels=False, duplicates='drop') + 1)
)

summary_sa_q = (
    sa_per_sub_test
    .groupby(['curriculum_label', 'sa_quantile', 'stimType'], as_index=False)
    .agg(mean_acc=('acc', 'mean'),
         se=('acc', lambda x: x.std(ddof=1) / np.sqrt(len(x))),
         n=('acc', 'count'))
)
summary_sa_q['ymin'] = summary_sa_q['mean_acc'] - summary_sa_q['se']
summary_sa_q['ymax'] = summary_sa_q['mean_acc'] + summary_sa_q['se']
# One combined group key so each (curriculum, stimType) line connects its own points
summary_sa_q['line_group'] = summary_sa_q['curriculum_label'] + ' | ' + summary_sa_q['stimType']


p_sa_test = (
    ggplot(summary_sa_q, aes(x='sa_quantile', y='mean_acc',
                             color='stimType', group='line_group'))
    + geom_line()
    + geom_point()
    + geom_errorbar(aes(ymin='ymin', ymax='ymax'), width=0.2)
    + geom_hline(yintercept=0.5, linetype='dotted', color='black', alpha=0.5)
    + scale_color_manual(values=stimColors)
    + scale_x_continuous(breaks=list(range(1, n_quantiles + 1)))
    + labs(x='Selective attention quintile',
           y='Test accuracy', color='Stim type',
           shape='Curriculum', linetype='Curriculum')
    + facet_wrap('~curriculum_label')
    + theme_classic()
    + theme(
        axis_text=element_text(size=7), 
        text=element_text(size=10), 
        legend_text=element_text(size=7), 
        figure_size=(2.5,2), 
        panel_spacing=0.02, 
        legend_position=(0.98, 0.1),
        legend_key_size=8,
        legend_key_spacing_y=2,
        legend_spacing=2,
    )
)

p_sa_test

p_sa_test.save(
    filename='analysis/figs/fig5C.svg',
    format="svg",
    verbose=False
)

## Supplementary

In [ ]:
sa_learning = per_sub_learning.merge(pretraining_sa.rename('selective_attention'), on='subjectId')
sa_learning['sa_group'] = (
    sa_learning
    .groupby('curriculum_label')['selective_attention']
    .transform(lambda x: np.where(x <= x.median(), 'below median', 'above median'))
)
n_quantiles = 3
sa_learning['sa_quantile'] = (
    sa_learning
    .groupby('curriculum_label')['selective_attention']
    .transform(lambda x: pd.qcut(x, q=n_quantiles, labels=False, duplicates='drop') + 1)
)
sa_learning['sa_quantile'] = sa_learning['sa_quantile'].astype(str)

# Across-subject summary: mean and standard error (SE)
summary = (
    sa_learning
    .groupby(['curriculum_label', 'sa_quantile', 'stimType', 'bin', 'bin_mid'], as_index=False)
    .agg(mean_acc=('acc', 'mean'), se=('acc', lambda x: x.std(ddof=1) / np.sqrt(len(x))))
)
#summary = summary[summary['stimType'] == 'Exception']
# Create ymin/ymax columns for errorbars
summary['ymin'] = summary['mean_acc'] - summary['se']
summary['ymax'] = summary['mean_acc'] + summary['se']

# Compute vertical lines every 48 trials (in trial number space)
max_trial = int(df_learning['totalTrialNum'].max())
vlines = list(range(48, max_trial, 48))
# Put vlines into a small DataFrame so plotnine treats them as continuous values
vline_df = pd.DataFrame({'x': vlines}) if len(vlines) > 0 else pd.DataFrame({'x': []})

# Build the plot
p = (
    ggplot(summary, aes(x='bin_mid', y='mean_acc', color='sa_quantile', shape='stimType'))
    + geom_line()
    + geom_point()
    + geom_errorbar(aes(ymin='ymin', ymax='ymax'), width=6)
    + (geom_vline(mapping=aes(xintercept='x'), data=vline_df, color='red', linetype='dashed', alpha=0.2) if not vline_df.empty else 0)
    + geom_hline(yintercept=0.5, linetype='solid', color='black', alpha=1)
    + labs(x='Trial', y='Accuracy', color='Selective attention', shape='Stimulus Type')
    + scale_x_continuous(limits=(0, max_trial))
    + scale_y_continuous(limits=(0, 1))
    # graded blues for low to high SA quantiles
    + scale_color_manual(values=["#8FBFD9","#4F93C0","#1F5A89"])
    + facet_wrap('~curriculum_label')
    + theme_classic()
    + theme(axis_text_x=element_text(rotation=0, ha='center'))
    + theme(axis_text=element_text(size=7), text=element_text(size=10), figure_size=(4.5,2.4), panel_spacing=0.03, legend_text=element_text(size=7),
        legend_key_height=5)

)

p

p.save(
    filename='analysis/figs/supp.svg',
    format="svg",
    verbose=False
)


In [17]:
df_learning_curriculum_smrt = df_learning[df_learning['curriculum'] == 'Simple To Complex'].merge(sa_learning[['subjectId', 'selective_attention', 'sa_group']], on='subjectId')
df_learning_control_smrt = df_learning[df_learning['curriculum'] == 'No Curriculum'].merge(sa_learning[['subjectId', 'selective_attention', 'sa_group']], on='subjectId')
df_learning_curriculum_smrt.to_csv('data/exp2_learning_curriculum_smrt.csv', index=False)
df_learning_control_smrt.to_csv('data/exp2_learning_control_smrt.csv', index=False)